# Chapter 13 — Abstract Manifolds, Tangent and Cotangent Bundles

Companion notebook to Chapter 13 of *Advanced Calculus for Machine Learning*.
Every numerical claim of the chapter is recomputed here with PyTorch and
checked with an assertion.

The rule that comes back in every section: under a change of chart with
Jacobian matrix `J`, columns (tangent vectors) become `J [v]`, rows
(covectors) become `[alpha] J^{-1}`, and a Hessian at a critical point
becomes `J^{-T} H J^{-1}`.

In [1]:
import math
import warnings
import torch
from torch.func import jacrev, jacfwd, hessian, vjp

warnings.filterwarnings("ignore", category=FutureWarning)
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
print(torch.__version__)
T = torch.tensor

2.15.0+cu132


## 1. Charts and transition maps

Example 13.6 (the lines of the plane), Example 13.7 (the circle `R/Z`),
Exercise 13.1 (the two stereographic charts of the circle) and the pitfall
on incompatible charts.

In [2]:
# lines of the plane: y = a x + b (chart 1) and x = a' y + b' (chart 2)
tau = lambda ab: torch.stack([1 / ab[0], -ab[1] / ab[0]])
ab = T([-0.8, 1.5])
a2 = tau(ab)
xs = torch.linspace(-2, 2, 9); ys = ab[0] * xs + ab[1]           # points of the line, from chart 1
assert torch.allclose(xs, a2[0] * ys + a2[1])                    # they satisfy the equation of chart 2
assert torch.allclose(tau(tau(ab)), ab)                          # the transition map is its own inverse
Jt = jacrev(tau)(ab)
print("transition (a, b) -> (1/a, -b/a); Jacobian determinant", round(torch.det(Jt).item(), 6), "= 1/a^3")
assert abs(torch.det(Jt).item() - 1 / ab[0].item() ** 3) < 1e-12

# the circle R/Z: two charts, transition = translation on each piece
phi1 = lambda x: x % 1.0                                         # values in (0, 1), misses [0]
phi2 = lambda x: (x + 0.5) % 1.0 - 0.5                           # values in (-1/2, 1/2), misses [1/2]
for x in (0.2, 0.7, 3.2, -1.3):
    u1, u2 = phi1(x), phi2(x)
    assert abs(u2 - (u1 if u1 < 0.5 else u1 - 1)) < 1e-12
    assert abs(phi1(x + 5) - u1) < 1e-12                         # well defined on classes
print("R/Z: transition x -> x on (0, 1/2) and x -> x - 1 on (1/2, 1)")

# Exercise 13.1: stereographic charts of the circle
th = torch.linspace(0.3, 6.0, 25)
th = th[(torch.sin(th).abs() < 0.999)]
x, y = torch.cos(th), torch.sin(th)
tN, tS = x / (1 - y), x / (1 + y)
assert torch.allclose(tN * tS, torch.ones_like(tN))
print("stereographic charts of the circle: phi_S = 1 / phi_N")

# incompatible charts on R: u -> u^(1/3) is not differentiable at 0
cuberoot = lambda u: torch.sign(u) * u.abs() ** (1 / 3)
slopes = [(cuberoot(T(h)) / h).item() for h in (1e-3, 1e-6, 1e-9)]
print("difference quotients of u^(1/3) at 0:", [f"{v:.0e}" for v in slopes])
assert slopes[2] > 1e4 * slopes[0]

transition (a, b) -> (1/a, -b/a); Jacobian determinant -1.953125 = 1/a^3
R/Z: transition x -> x on (0, 1/2) and x -> x - 1 on (1/2, 1)
stereographic charts of the circle: phi_S = 1 / phi_N
difference quotients of u^(1/3) at 0: ['1e+02', '1e+04', '1e+06']


## 2. Tangent vectors

Definition 13.9, Theorem 13.10 and Proposition 13.12 on the sphere `S^2`
with its two stereographic charts: two curves that are tangent in one chart
are tangent in the other; the columns of components are related by the
Jacobian matrix of the transition map; and the abstract tangent vector
corresponds to the velocity in `R^3`.

In [3]:
phiN = lambda q: q[:2] / (1 - q[2])
phiS = lambda q: q[:2] / (1 + q[2])
psiN = lambda t: torch.cat([2 * t, (t @ t - 1).reshape(1)]) / (t @ t + 1)
psiS = lambda t: torch.cat([2 * t, (1 - t @ t).reshape(1)]) / (t @ t + 1)
tauNS = lambda t: t / (t @ t)                                    # phi_S o phi_N^{-1}
unit = lambda q: q / q.norm()

p = unit(T([1.0, 2.0, 2.0]))
w = T([0.5, -1.0, 0.75]); w = w - (w @ p) * p                     # tangent to the sphere at p
z = T([0.3, 0.2, -1.0])
c1 = lambda s: unit(p + s * w)
c2 = lambda s: unit(p + s * w + s ** 2 * z)                       # same velocity, another curve
c3 = lambda s: unit(p + s * (w + 0.1 * z))                        # another velocity
s0 = T(0.0)
theta = lambda chart, c: jacfwd(lambda s: chart(c(s)))(s0)
aN1, aN2, aN3 = theta(phiN, c1), theta(phiN, c2), theta(phiN, c3)
aS1, aS2, aS3 = theta(phiS, c1), theta(phiS, c2), theta(phiS, c3)
print("components in the north chart:", [round(v, 4) for v in aN1.tolist()],
      "  in the south chart:", [round(v, 4) for v in aS1.tolist()])
assert torch.allclose(aN1, aN2) and torch.allclose(aS1, aS2)      # tangent in one chart <=> in the other
assert not torch.allclose(aN1, aN3) and not torch.allclose(aS1, aS3)
J = jacrev(tauNS)(phiN(p))
assert torch.allclose(aS1, J @ aN1) and torch.allclose(aS3, J @ aN3)   # theta' = D tau . theta
# Proposition 13.12: the velocity in R^3 is D psi(t) theta
assert torch.allclose(jacfwd(psiN)(phiN(p)) @ aN1, jacfwd(c1)(s0))
assert torch.allclose(jacfwd(psiS)(phiS(p)) @ aS1, jacfwd(c1)(s0))
assert torch.allclose(jacfwd(c1)(s0), w)
print("[v]' = J [v]; D psi(t) [v] is the velocity in R^3, the same from both charts")

components in the north chart: [3.75, 1.5]   in the south chart: [0.21, -0.78]
[v]' = J [v]; D psi(t) [v] is the velocity in R^3, the same from both charts


## 3. The differential of a map

Proposition 13.14: the matrix of `DF(p)` in coordinate bases is the Jacobian
matrix of the local representative; another pair of charts gives
`J_out . (matrix) . J_in^{-1}`. Example 13.16: `R/Z` and the unit circle.

In [4]:
A = T([[1.0, 0.3, 0.0], [-0.2, 1.1, 0.4], [0.1, 0.0, 0.9]])
F = lambda q: unit(A @ q)                                         # a smooth map S^2 -> S^2
FN = lambda t: phiN(F(psiN(t)))                                   # local representative, north charts
FS = lambda t: phiS(F(psiS(t)))                                   # local representative, south charts
t = phiN(p); tS = phiS(p)
JN, JS = jacrev(FN)(t), jacrev(FS)(tS)
J_in = jacrev(tauNS)(t); J_out = jacrev(tauNS)(FN(t))
assert torch.allclose(JS, J_out @ JN @ torch.linalg.inv(J_in))
# push-forward of the tangent vector of c1: class of F o c1
assert torch.allclose(theta(phiN, lambda s: F(c1(s))), JN @ aN1)
# chain rule for G o F
G = lambda q: unit(q + T([0.2, 0.0, 0.1]) * q[0])
GN = lambda t: phiN(G(psiN(t)))
assert torch.allclose(jacrev(lambda t: GN(FN(t)))(t), jacrev(GN)(FN(t)) @ JN)
print("matrix of DF(p) in the north charts:\n", JN)

# Example 13.16: [x] -> (cos 2 pi x, sin 2 pi x)
E = lambda x: torch.stack([torch.cos(2 * math.pi * x), torch.sin(2 * math.pi * x)])
for x in (0.1, 0.45, 0.8):
    v = jacfwd(E)(T(x))
    assert torch.allclose(v, 2 * math.pi * T([-math.sin(2 * math.pi * x), math.cos(2 * math.pi * x)]))
    assert abs(v.norm().item() - 2 * math.pi) < 1e-12
assert torch.allclose(E(T(0.3)), E(T(1.3)))
print("R/Z -> S^1: well defined, velocity of norm 2 pi, never zero")

matrix of DF(p) in the north charts:
 tensor([[ 0.5027, -0.1497],
        [ 0.0387,  0.4842]])
R/Z -> S^1: well defined, velocity of norm 2 pi, never zero


## 4. Cotangent vectors and the change of chart

Proposition 13.18, Theorem 13.19 and Table 13.1, on Example 13.20 (polar
coordinates) and Exercise 13.4.

In [5]:
cart = lambda rt: torch.stack([rt[0] * torch.cos(rt[1]), rt[0] * torch.sin(rt[1])])   # (r, theta) -> (x, y)
rt = T([2.0, math.pi / 6])
J = jacrev(cart)(rt)                                              # Jacobian of polar -> Cartesian
r, thv = rt.tolist()
assert torch.allclose(J, T([[math.cos(thv), -r * math.sin(thv)], [math.sin(thv), r * math.cos(thv)]]))
# f = x: row of partial derivatives in the polar chart, and polar components of the vector d/dx
f_polar = lambda rt: cart(rt)[0]
row = jacrev(f_polar)(rt)
col = torch.linalg.solve(J, T([1.0, 0.0]))
print("[dx] in the polar chart:", [round(v, 4) for v in row.tolist()],
      "   polar components of d/dx:", [round(v, 4) for v in col.tolist()])
assert torch.allclose(row, T([math.cos(thv), -r * math.sin(thv)]))
assert torch.allclose(col, T([math.cos(thv), -math.sin(thv) / r]))
assert not torch.allclose(row, col)                               # the transposed row is not the vector
assert abs((row @ col).item() - 1.0) < 1e-12                      # dx(d/dx) = 1 in the polar chart too
rt1 = T([1.0, 0.9])                                               # on the circle r = 1 they agree
assert torch.allclose(jacrev(f_polar)(rt1), torch.linalg.solve(jacrev(cart)(rt1), T([1.0, 0.0])))

# Exercise 13.4
v_pol = torch.linalg.solve(J, T([0.0, 1.0])); a_pol = J[1]
print("Exercise 13.4: d/dy ->", [round(v, 4) for v in v_pol.tolist()], "  dy ->", [round(v, 4) for v in a_pol.tolist()])
assert torch.allclose(v_pol, T([0.5, math.sqrt(3) / 4])) and torch.allclose(a_pol, T([0.5, math.sqrt(3)]))
assert abs((a_pol @ v_pol).item() - 1) < 1e-12

# rows and columns, for a random function, vector and chart change (north -> south on the sphere)
h = lambda q: torch.sin(q[0]) + q[1] * q[2]
hN = lambda t: h(psiN(t)); hS = lambda t: h(psiS(t))
Jns = jacrev(tauNS)(t)
rowN, rowS = jacrev(hN)(t), jacrev(hS)(tS)
assert torch.allclose(rowS, rowN @ torch.linalg.inv(Jns))         # [alpha]' = [alpha] J^{-1}
assert abs((rowN @ aN1 - rowS @ aS1).item()) < 1e-12              # the pairing does not depend on the chart
assert abs((rowN @ aN1).item() - jacfwd(lambda s: h(c1(s)))(s0).item()) < 1e-12
print("dh(v) =", round((rowN @ aN1).item(), 6), "in both charts = (h o c)'(0)")

[dx] in the polar chart: [0.866, -1.0]    polar components of d/dx: [0.866, -0.25]
Exercise 13.4: d/dy -> [0.5, 0.433]   dy -> [0.5, 1.7321]
dh(v) = 0.305812 in both charts = (h o c)'(0)


## 5. The Hessian at a critical point

Proposition 13.22 and Exercise 13.8: at a critical point the matrix of
second partial derivatives changes by congruence; its eigenvalues change,
its signature does not. At a point that is not critical the rule fails.

In [6]:
to_xy = lambda uv: torch.stack([uv[0] - uv[1] - uv[1] ** 2, uv[1]])       # inverse of (u, v) = (x + y + y^2, y)
to_uv = lambda xy: torch.stack([xy[0] + xy[1] + xy[1] ** 2, xy[1]])
O = torch.zeros(2)
J = jacrev(to_uv)(O); Ji = torch.linalg.inv(J)
f = lambda xy: xy[0] ** 2 - xy[1] ** 2
H, Hp = hessian(f)(O), hessian(lambda uv: f(to_xy(uv)))(O)
print("H =", H.tolist(), "  H' =", Hp.tolist())
assert torch.allclose(Hp, Ji.T @ H @ Ji) and torch.allclose(Hp, T([[2.0, -2.0], [-2.0, 0.0]]))
e, ep = torch.linalg.eigvalsh(H), torch.linalg.eigvalsh(Hp)
print("eigenvalues:", e.tolist(), "and", [round(v, 4) for v in ep.tolist()])
assert torch.allclose(ep, T([1 - math.sqrt(5), 1 + math.sqrt(5)]))
assert (e > 0).sum() == (ep > 0).sum() == 1 and (e < 0).sum() == (ep < 0).sum() == 1
# along a curve through the critical point: (f o c)''(0) = H(v, v), whatever the curve
cq = lambda s: T([0.4, -0.7]) * s + T([1.0, 2.0]) * s ** 2
d2 = jacfwd(jacfwd(lambda s: f(cq(s))))(T(0.0)).item()
vq = T([0.4, -0.7])
assert abs(d2 - (vq @ H @ vq).item()) < 1e-12
# not critical: f = x
g = lambda xy: xy[0]
Hg, Hgp = hessian(g)(O), hessian(lambda uv: g(to_xy(uv)))(O)
print("f = x: second derivatives", Hg.tolist(), "in (x, y) and", (Hgp + 0.0).tolist(), "in (u, v)")
assert Hg.abs().max() == 0 and torch.allclose(Hgp, T([[0.0, 0.0], [0.0, -2.0]]))

H = [[2.0, 0.0], [0.0, -2.0]]   H' = [[2.0, -2.0], [-2.0, 0.0]]
eigenvalues: [-2.0, 2.0] and [-1.2361, 3.2361]
f = x: second derivatives [[0.0, 0.0], [0.0, 0.0]] in (x, y) and [[0.0, 0.0], [0.0, -2.0]] in (u, v)


## 6. Push-forward and pull-back

Proposition 13.26 with Exercise 13.7, the pitfall on vector fields, and
Exercise 13.6 (the doubling map of the circle).

In [7]:
Fc = lambda t: torch.stack([t ** 2, t ** 3])
beta = lambda xy: torch.stack([torch.zeros(()), xy[0]])            # the covector field x dy on R^2, as a row
for tv in (-1.5, 0.0, 0.7, 2.0):
    tt = T(tv)
    pulled = beta(Fc(tt)) @ jacfwd(Fc)(tt)                         # row . Jacobian
    _, back = vjp(Fc, tt)                                          # the same product, by reverse mode
    assert abs(pulled.item() - 3 * tv ** 4) < 1e-12 and abs(back(beta(Fc(tt)))[0].item() - pulled.item()) < 1e-12
    assert torch.allclose(jacfwd(Fc)(tt), T([2 * tv, 3 * tv ** 2]))
print("F*(x dy) = 3 t^4 dt;  DF(t) d/dt = 2t d/dx + 3t^2 d/dy, zero at t = 0 only")

sq = lambda t: t ** 2
print("push-forward of d/dt by t -> t^2 at the point 1:", jacfwd(sq)(T(1.0)).item(), "from t = 1 and",
      jacfwd(sq)(T(-1.0)).item(), "from t = -1")
assert jacfwd(sq)(T(1.0)) == 2 and jacfwd(sq)(T(-1.0)) == -2

# doubling map on R/Z in the two charts: local representatives x -> 2x - m
for x in (0.1, 0.3, 0.6, 0.9):
    assert abs(jacfwd(lambda u: 2 * u)(T(x)).item() - 2) < 1e-12
    assert abs(phi1(2 * x) - (2 * x - math.floor(2 * x))) < 1e-12
assert abs(phi1(2 * 0.0) - phi1(2 * 0.5)) < 1e-12
print("doubling map: derivative 2 in the charts, F[0] = F[1/2]")

F*(x dy) = 3 t^4 dt;  DF(t) d/dt = 2t d/dx + 3t^2 d/dy, zero at t = 0 only
push-forward of d/dt by t -> t^2 at the point 1: 2.0 from t = 1 and -2.0 from t = -1
doubling map: derivative 2 in the charts, F[0] = F[1/2]


## 7. Machine learning: one loss, three charts

The first box of Section 13.6, Figure 13.4 and Exercise 13.5: the Gaussian
family with the charts `(mu, sigma)`, `(mu, ln sigma)` and
`(mu, sigma^2)`.

In [8]:
L = lambda ms: torch.log(ms[1]) + (1 + ms[0] ** 2) / (2 * ms[1] ** 2)     # in the chart (mu, sigma)
charts = {
    "(mu, sigma)":    (lambda x: x,                                    lambda ms: ms),
    "(mu, ln sigma)": (lambda x: torch.stack([x[0], torch.exp(x[1])]), lambda ms: torch.stack([ms[0], torch.log(ms[1])])),
    "(mu, sigma^2)":  (lambda x: torch.stack([x[0], torch.sqrt(x[1])]), lambda ms: torch.stack([ms[0], ms[1] ** 2])),
}
ms0 = T([1.0, 2.0]); eta = 1.2
rows, landed = {}, {}
for name, (to_ms, from_ms) in charts.items():
    x0 = from_ms(ms0)
    rows[name] = jacrev(lambda x: L(to_ms(x)))(x0)
    landed[name] = to_ms(x0 - eta * rows[name])
    print(f"{name:15s} [dL] = {[round(v, 4) for v in rows[name].tolist()]}   after one step: (mu, sigma) = "
          f"{[round(v, 4) for v in landed[name].tolist()]}")
assert torch.allclose(rows["(mu, sigma)"], T([0.25, 0.25]))
assert torch.allclose(rows["(mu, ln sigma)"], T([0.25, 0.5])) and torch.allclose(rows["(mu, sigma^2)"], T([0.25, 0.0625]))
assert torch.allclose(landed["(mu, sigma)"], T([0.7, 1.7]))
assert torch.allclose(landed["(mu, ln sigma)"], T([0.7, 2 * math.exp(-0.6)]))
assert torch.allclose(landed["(mu, sigma^2)"], T([0.7, math.sqrt(3.925)]))
# rows are related by [dL]' = [dL] J^{-1}
for name, (to_ms, from_ms) in charts.items():
    Jc = jacrev(from_ms)(ms0)
    assert torch.allclose(rows[name], rows["(mu, sigma)"] @ torch.linalg.inv(Jc))
# first-order formula: the step of the chart x', seen from x, is -eta (J^T J)^{-1} [dL]^T + O(eta^2)
to_ms, from_ms = charts["(mu, ln sigma)"]
Jc = jacrev(from_ms)(ms0)
pre = torch.linalg.inv(Jc.T @ Jc)
assert torch.allclose(pre, torch.diag(T([1.0, 4.0])))
errs = []
for e_ in (1.2, 0.12, 0.012):
    exact = to_ms(from_ms(ms0) - e_ * rows["(mu, ln sigma)"]) - ms0
    errs.append((exact - (-e_ * pre @ rows["(mu, sigma)"])).norm().item())
print("error of the first-order formula for eta = 1.2, 0.12, 0.012:", [f"{v:.2e}" for v in errs])
assert abs(errs[0] - 2 * (math.exp(-0.6) - 1 + 0.6)) < 1e-12 and 80 < errs[1] / errs[2] < 120

# what does not depend on the chart: the critical point and the signature of the Hessian there
crit = T([0.0, 1.0])
for name, (to_ms, from_ms) in charts.items():
    xc = from_ms(crit)
    assert jacrev(lambda x: L(to_ms(x)))(xc).abs().max() < 1e-12
    ev = torch.linalg.eigvalsh(hessian(lambda x: L(to_ms(x)))(xc))
    print(f"{name:15s} eigenvalues of the Hessian at the minimum: {[round(v, 4) for v in ev.tolist()]}")
    assert (ev > 0).all()
assert torch.allclose(torch.linalg.eigvalsh(hessian(lambda x: L(charts['(mu, sigma^2)'][0](x)))(T([0.0, 1.0]))), T([0.5, 1.0]))

# a map that is not a chart: w = u^3 creates a critical point
loss_w = lambda w_: (w_ - 1) ** 2
print("dL/dw at w = 0:", jacrev(loss_w)(T(0.0)).item(), "   d/du L(u^3) at u = 0:", jacrev(lambda u: loss_w(u ** 3))(T(0.0)).item() + 0.0)
assert jacrev(loss_w)(T(0.0)) == -2 and jacrev(lambda u: loss_w(u ** 3))(T(0.0)) == 0

(mu, sigma)     [dL] = [0.25, 0.25]   after one step: (mu, sigma) = [0.7, 1.7]
(mu, ln sigma)  [dL] = [0.25, 0.5]   after one step: (mu, sigma) = [0.7, 1.0976]
(mu, sigma^2)   [dL] = [0.25, 0.0625]   after one step: (mu, sigma) = [0.7, 1.9812]
error of the first-order formula for eta = 1.2, 0.12, 0.012: ['2.98e-01', '3.53e-03', '3.59e-05']
(mu, sigma)     eigenvalues of the Hessian at the minimum: [1.0, 2.0]
(mu, ln sigma)  eigenvalues of the Hessian at the minimum: [1.0, 2.0]
(mu, sigma^2)   eigenvalues of the Hessian at the minimum: [0.5, 1.0]
dL/dw at w = 0: -2.0    d/du L(u^3) at u = 0: 0.0


## 8. Machine learning: symmetries

The second box of Section 13.6, Exercise 13.9 and Exercise 13.3: the
differential of an invariant loss vanishes along the symmetry, a quantity
is conserved by the gradient flow, and the Hessian at a minimum is
degenerate.

In [9]:
# Exercise 13.9
L2 = lambda w_: 0.5 * (w_[1] * w_[0] - 3) ** 2
w0 = T([2.0, 0.5]); eta = 0.1
g = jacrev(L2)(w0)
print("gradient at (2, 0.5):", g.tolist(), "   w1 g1 =", (w0[0] * g[0]).item(), "  w2 g2 =", (w0[1] * g[1]).item())
assert torch.allclose(g, T([-1.0, -4.0])) and abs((w0[0] * g[0] - w0[1] * g[1]).item()) < 1e-12
w1 = w0 - eta * g
Q = lambda w_: (w_[0] ** 2 - w_[1] ** 2).item()
print("w1^2 - w2^2 before and after one step:", Q(w0), round(Q(w1), 6), "  eta^2 (g1^2 - g2^2) =",
      round(eta ** 2 * (g[0] ** 2 - g[1] ** 2).item(), 6))
assert abs(Q(w1) - Q(w0) - eta ** 2 * (g[0] ** 2 - g[1] ** 2).item()) < 1e-12 and abs(Q(w1) - 3.6) < 1e-12
# gradient flow, approximated by small steps: the drift of w1^2 - w2^2 is proportional to the step
drift = []
for h_ in (1e-2, 1e-3):
    wv = w0.clone()
    for _ in range(int(1.0 / h_)):
        wv = wv - h_ * jacrev(L2)(wv)
    drift.append(abs(Q(wv) - Q(w0)))
print("drift of w1^2 - w2^2 over a unit time, steps 1e-2 and 1e-3:", [f"{v:.2e}" for v in drift])
assert 8 < drift[0] / drift[1] < 12
# Hessian at a minimum: rank 1, kernel along the symmetry
wm = T([1.5, 2.0])
Hm = hessian(L2)(wm)
assert torch.allclose(Hm, T([[4.0, 3.0], [3.0, 2.25]])) and (Hm @ T([1.5, -2.0])).abs().max() < 1e-12
assert torch.linalg.matrix_rank(Hm) == 1
print("Hessian at the minimum (1.5, 2):", Hm.tolist(), " kernel spanned by (w1, -w2)")

# a ReLU network with one hidden layer: scaling the weights into and out of a hidden unit
torch.manual_seed(5)
X, Y = torch.randn(40, 3), torch.randn(40, 2)
def net_loss(W1, b1, W2):
    return ((torch.relu(X @ W1.T + b1) @ W2.T - Y) ** 2).mean()
W1, b1, W2 = torch.randn(4, 3), torch.randn(4), torch.randn(2, 4)
c = 2.5; j = 1
W1c, b1c, W2c = W1.clone(), b1.clone(), W2.clone()
W1c[j] *= c; b1c[j] *= c; W2c[:, j] /= c
assert abs((net_loss(W1, b1, W2) - net_loss(W1c, b1c, W2c)).item()) < 1e-12        # same function
g1, gb, g2 = jacrev(net_loss, argnums=(0, 1, 2))(W1, b1, W2)
pair = (g1[j] @ W1[j] + gb[j] * b1[j] - g2[:, j] @ W2[:, j]).item()                  # dL(xi), xi = (in, -out)
print("dL along the scaling direction of hidden unit", j, ":", f"{pair:.1e}")
assert abs(pair) < 1e-12
bal = lambda A_, b_, B_: ((A_ ** 2).sum(1) + b_ ** 2 - (B_ ** 2).sum(0))             # per hidden unit
before = bal(W1, b1, W2); step = 1e-3
after = bal(W1 - step * g1, b1 - step * gb, W2 - step * g2)
print("change of |in|^2 - |out|^2 per hidden unit after a step of 1e-3:", [f"{v:.1e}" for v in (after - before).tolist()])
assert (after - before).abs().max() < 10 * step ** 2 * max(g1.norm().item(), g2.norm().item()) ** 2

# Exercise 13.3: from parameters (alpha, beta, gamma) to the line, in the chart (a, b)
Qmap = lambda abc: torch.stack([-abc[0] / abc[1], -abc[2] / abc[1]])
abc = T([0.7, -1.3, 2.0])
JQ = jacrev(Qmap)(abc)
assert torch.allclose(JQ, T([[-1 / abc[1], abc[0] / abc[1] ** 2, 0.0], [0.0, abc[2] / abc[1] ** 2, -1 / abc[1]]]))
assert torch.linalg.matrix_rank(JQ) == 2 and (JQ @ abc).abs().max() < 1e-12
assert torch.allclose(Qmap(abc), Qmap(-3.0 * abc))
print("parameters -> line: a submersion, kernel spanned by (alpha, beta, gamma)")

gradient at (2, 0.5): [-1.0, -4.0]    w1 g1 = -2.0   w2 g2 = -2.0
w1^2 - w2^2 before and after one step: 3.75 3.6   eta^2 (g1^2 - g2^2) = -0.15


drift of w1^2 - w2^2 over a unit time, steps 1e-2 and 1e-3: ['1.55e-02', '1.51e-03']
Hessian at the minimum (1.5, 2): [[4.0, 3.0], [3.0, 2.25]]  kernel spanned by (w1, -w2)
dL along the scaling direction of hidden unit 1 : 0.0e+00
change of |in|^2 - |out|^2 per hidden unit after a step of 1e-3: ['9.2e-05', '-2.1e-06', '-2.8e-05', '6.8e-07']
parameters -> line: a submersion, kernel spanned by (alpha, beta, gamma)


## Summary

* Transition maps between charts are diffeomorphisms; on `R/Z` they are
  translations.
* Tangency of curves, the pairing of a covector with a vector, the critical
  points of a function and the signature of its Hessian there do not depend
  on the chart.
* Components do: columns change by `J`, rows by `J^{-1}`, Hessians at
  critical points by congruence.
* Gradient descent run in three charts of the same family of distributions
  takes three different steps for the same covector `dL`.
* A symmetry of the parameters gives a direction on which `dL` vanishes, a
  conserved quantity of the gradient flow and a degenerate Hessian.